# 04 - Ingestão Bronze: Detecção e Carga Incremental de Clientes
**Squad 2 — Real Time for Business | Dupla 2**  
**Integrantes:** Kauan & Leandro  
**Tabelas de Escopo:** `ecommerce_clientes` (arquivos Parquet no ADLS Gen2)  
**Branch:** `feat/squad2/kauan-leandro`  

### Objetivo da Task (Bronze — ecommerce_clientes):
1. **Leitura da Origem:** Ler arquivos `ecommerce_clientes.parquet` do ADLS Gen2 (container `raw`, path `real-time-data/`).
2. **Detecção Incremental:** Identificar arquivos novos via anti-join com o log de controle (arquivos já processados).
3. **Persistência Bronze:** Gravar no ADLS Delta (append) com `bronze_ingested_at`, particionado por ano/mes/dia/hora.
4. **Log de Controle:** Registrar cada arquivo processado para garantir idempotência.
5. **Monitoramento:** Loop de verificacão a cada 15 segundos (max 90s) com tolerância a falhas.

In [0]:
%run ./00_setup_config.ipynb

In [0]:
# Importação das bibliotecas necessárias

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from dotenv import load_dotenv
from pathlib import Path
from pyspark.sql import functions as F
import os

In [0]:
# Localiza e carrega o arquivo .env com as credenciais do projeto

env_path = next(Path("/Workspace").rglob(".env"), None)

if env_path:
    load_dotenv(env_path)
    print(".env carregado com sucesso.")
else:
    print(".env não encontrado.")

In [0]:
# Cria a credencial para autenticar o acesso ao Azure

credential = ClientSecretCredential(
    tenant_id=os.getenv("ADLS_TENANT_ID"),
    client_id=os.getenv("ADLS_CLIENT_ID"),
    client_secret=os.getenv("ADLS_CLIENT_SECRET")
)

print("Credencial do Azure criada com sucesso.")

In [0]:
# Cria a conexão com a conta e o container do Data Lake

storage_account_name = "internshipdatalake"
container_name = "raw"

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)

fs_client = service_client.get_file_system_client(container_name)

print("Conexão com o ADLS criada com sucesso.")

In [0]:
# Localiza os arquivos Parquet da ecommerce_clientes no Data Lake

arquivos_encontrados = []

for arquivo in fs_client.get_paths(path="real-time-data", recursive=True):
    if not arquivo.is_directory and arquivo.name.endswith("ecommerce_clientes.parquet"):
        arquivos_encontrados.append(arquivo.name)

print(f"Arquivos encontrados: {len(arquivos_encontrados)}")

for arquivo in arquivos_encontrados:
    print(arquivo)

In [0]:
# Prepara as credenciais OAuth para o Spark acessar o ADLS

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": (
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token"
    )
}

print("Credenciais OAuth do Spark preparadas.")

In [0]:
adls_tenant_id = os.getenv("ADLS_TENANT_ID")
adls_client_id = os.getenv("ADLS_CLIENT_ID")
adls_client_secret = os.getenv("ADLS_CLIENT_SECRET")
adls_storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")

import pandas as pd
from io import BytesIO

credential = ClientSecretCredential(adls_tenant_id, adls_client_id, adls_client_secret)
service_client = DataLakeServiceClient(
    account_url=f"https://{adls_storage_account}.dfs.core.windows.net/",
    credential=credential,
)
file_system_client = service_client.get_file_system_client(file_system="raw")

print("Conexão via SDK pronta (file_system_client).")

In [0]:

caminho_bronze = (
    f"abfss://squad2@{storage_account_name}.dfs.core.windows.net/"
    "bronze/ecommerce_clientes")
caminho_control_log = f"abfss://squad2@{storage_account_name}.dfs.core.windows.net/grupo2/metadata/ingestion_control_log"

print(f"Caminho Bronze: {caminho_bronze}")
print(f"Caminho log de controle: {caminho_control_log}")

In [0]:
def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

print("Função de verificação de tabela Delta pronta.")

In [0]:
def detectar_novos_clientes():
    todos_arquivos = [
        p.name for p in file_system_client.get_paths(path="real-time-data", recursive=True)
        if p.name.endswith("ecommerce_clientes.parquet")
    ]

    if tabela_delta_existe(caminho_control_log):
        df_control = spark.read.format("delta").options(**adls_options).load(caminho_control_log)
        ja_processados = {
            r["arquivo_processado"] for r in
            df_control.select("arquivo_processado").collect()
        }
    else:
        ja_processados = set()

    arquivos_novos = [a for a in todos_arquivos if a not in ja_processados]

    if not arquivos_novos:
        return None, []

    dfs = []
    for caminho in arquivos_novos:
        file_client = file_system_client.get_file_client(caminho)
        conteudo = file_client.download_file().readall()
        pdf = pd.read_parquet(BytesIO(conteudo))
        pdf["bronze_source_file"] = caminho 
        dfs.append(pdf)

    pdf_final = pd.concat(dfs, ignore_index=True)
    df_novos = spark.createDataFrame(pdf_final)

    return df_novos, arquivos_novos

print("Função de detecção de arquivos novos pronta.")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import current_timestamp, year, month, dayofmonth, hour, col, count
from datetime import datetime

schema_log = StructType([
    StructField("tabela_origem", StringType(), False),
    StructField("arquivo_processado", StringType(), False),
    StructField("total_linhas", LongType(), False),
    StructField("status", StringType(), False),
    StructField("timestamp_processamento", StringType(), False),
])

def processar_bronze(df_novos, arquivos_novos):
    if df_novos is None:
        print("    Nenhum arquivo novo para 'ecommerce_clientes'.")
        return

    total = df_novos.count()
    print(f"    >>> {len(arquivos_novos)} arquivo(s) novo(s) detectado(s) ({total} linhas)")

    df_bronze = (df_novos
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("ano", year(col("bronze_ingested_at")))
        .withColumn("mes", month(col("bronze_ingested_at")))
        .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
        .withColumn("hora", hour(col("bronze_ingested_at"))))

    (df_bronze.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .partitionBy("ano", "mes", "dia", "hora")
        .save(caminho_bronze))

    print("    [OK] Bronze gravada.")

    try:
        resumo_por_arquivo = (df_novos
            .groupBy("bronze_source_file")
            .agg(count("*").alias("linhas"))
            .collect())

        agora = str(datetime.now())
        log_data = [("ecommerce_clientes", r["bronze_source_file"], int(r["linhas"]), "SUCCESS", agora) for r in resumo_por_arquivo]
        df_log = spark.createDataFrame(log_data, schema=schema_log)

        (df_log.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminho_control_log))

        print("    [OK] Log de controle atualizado.")
    except Exception as e_log:
        print(f"    [ERRO NO LOG] {type(e_log).__name__}: {e_log}")

print("Função de gravação na Bronze pronta.")

In [0]:
import time
from datetime import datetime, timedelta

intervalo_segundos = 15
duracao_maxima_segundos = 90
inicio = datetime.now()
fim_previsto = inicio + timedelta(seconds=duracao_maxima_segundos)
falhas_consecutivas = 0
MAX_FALHAS_CONSECUTIVAS = 5

print(f"Iniciando monitoramento às {inicio.strftime('%H:%M:%S')}.")
print(f"Vai rodar até {fim_previsto.strftime('%H:%M:%S')} (ou até interromper manualmente).\n")

while datetime.now() < fim_previsto:
    try:
        print(f"--- Verificação às {datetime.now().strftime('%H:%M:%S')} ---")
        df_novos, arquivos_novos = detectar_novos_clientes()
        processar_bronze(df_novos, arquivos_novos)
        falhas_consecutivas = 0
    except KeyboardInterrupt:
        print("\nMonitoramento interrompido manualmente.")
        break
    except Exception as e:
        falhas_consecutivas += 1
        print(f"    [ERRO] Falha na verificação ({falhas_consecutivas}/{MAX_FALHAS_CONSECUTIVAS}): {e}")
        if falhas_consecutivas >= MAX_FALHAS_CONSECUTIVAS:
            print(f"    {MAX_FALHAS_CONSECUTIVAS} falhas seguidas — interrompendo o loop.")
            break
    time.sleep(intervalo_segundos)

print(f"\nMonitoramento finalizado às {datetime.now().strftime('%H:%M:%S')}.")